# Extract Mispronunciations
To prepare for evaluation, specifically extracting False positive and False Negative rates, we must first establish ground truth labels for our L2-speaker data compared to a canonical source. To do this, we first align the phonemes of the 

# Imports

In [1]:
from huggingface_hub import login, logout
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC, AutoProcessor, AutoModelForCTC, pipeline
import datasets
from datasets import load_dataset, Audio
import torch

/home/peter/miniconda3/envs/thesis/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys, pathlib

PROJECT_ROOT = pathlib.Path('.').resolve()
while not (PROJECT_ROOT / '.git').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.eval.eval import (
    align_phones,
    phone_error_rate,
    get_canonical_labels,
    evaluate_mispronunciation_detection,
    OP_LABEL,
)

# Phoneme Collapsing

Map surface forms from ASR output or gold annotations to a normalised form before alignment.
This prevents spurious mismatches caused by annotation-level differences (e.g. velarisation
diacritics present in one source but not the other) rather than genuine mispronunciations.

## Fetch and inspect model Vocabs

In [3]:
en_model_id = "duck-hug-567/xls-r-300m-engphon-base"
ga_model_id = "duck-hug-567/xls-r-300m-gaphon-full-v2"

In [4]:
en_pipe = pipeline("automatic-speech-recognition", model=en_model_id)
en_processor = AutoProcessor.from_pretrained(en_model_id)

ga_pipe = pipeline("automatic-speech-recognition", model=ga_model_id)
ga_processor = AutoProcessor.from_pretrained(ga_model_id)

Loading weights: 100%|██████████| 424/424 [00:00<00:00, 793.04it/s, Materializing param=wav2vec2.masked_spec_embed]                                            


In [5]:
en_DICTIONARY = en_processor.tokenizer.get_vocab()
ga_DICTIONARY = ga_processor.tokenizer.get_vocab()

In [6]:
en_charset = en_DICTIONARY.keys()
ga_charset = ga_DICTIONARY.keys()

In [7]:
en_charset

dict_keys(['[PAD]', '[UNK]', 'aɪ', 'aʊ', 'b', 'd', 'eɪ', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'l̩', 'm', 'm̩', 'n', 'n̩', 'oʊ', 'p', 's', 't', 'u', 'v', 'w', 'z', '|', 'æ', 'ð', 'ŋ', 'ŋ̍', 'ɑ', 'ɔ', 'ɔɪ', 'ə', 'ə̥', 'ɚ', 'ɛ', 'ɝ', 'ɦ', 'ɨ', 'ɪ', 'ɹ', 'ɾ', 'ɾ̃', 'ʃ', 'ʉ', 'ʊ', 'ʌ', 'ʒ', 'ʔ', 'ʤ', 'ʧ', 'θ', '<s>', '</s>'])

In [8]:
print([phon for phon in ga_charset if len(phon) > 1])

['[PAD]', '[UNK]', 'ai', 'au', 'aː', 'bʲ', 'bˠ', 'dʲ', 'd̪ˠ', 'eː', 'fʲ', 'fˠ', 'hʲ', 'ia', 'iː', 'iˑə', 'lʲ', 'l̻̊ˠ', 'l̥ʲ', 'l̻ʲ', 'l̻ˠ', 'mʲ', 'mˠ', 'm̥ʲ', 'm̥ˠ', 'nʲ', 'nˠ', 'n̻̊ˠ', 'n̥ʲ', 'n̻ʲ', 'n̻ˠ', 'oː', 'pʲ', 'pˠ', 'sˠ', 'tʲ', 't̪', 't̪ˠ', 'ua', 'uː', 'uˑə', 'vʲ', 'vˠ', 'zʲ', 'zˠ', 'ɲ̊', 'ɾʲ', 'ɾˠ', 'ɾ̥ʲ', 'ɾ̥ˠ', '<s>', '</s>']


## Mapping

In [10]:
# Collapse smoke test
# Checks three things:
#   1. A mapped phone is replaced with its target
#   2. An unmapped phone passes through unchanged
#   3. A mixed sequence handles both correctly

from scripts.data_handling.collapse_phonemes import collapse_phones

# One example from each category in the map
assert collapse_phones(['bˠ'])   == ['b'], "velarized b -> b"
assert collapse_phones(['l̻ˠ'])   == ['l'], "laminal velarized l -> l"
assert collapse_phones(['ə̥'])    == ['ə'], "devoiced schwa -> schwa"

# Unmapped phone passes through unchanged
assert collapse_phones(['ʃ'])    == ['ʃ'], "unmapped phone unchanged"

# Mixed sequence
assert collapse_phones(['bˠ', 'a', 'l̻ˠ', 'ʃ']) == ['b', 'a', 'l', 'ʃ']

# Empty input
assert collapse_phones([]) == []

print("Collapse smoke test passed")


Collapse smoke test passed


# Evaluation

For each canonical phone position, we independently align:
- **gold** annotation → canonical → ground truth label (was it actually mispronounced?)
- **ASR output** → canonical → predicted label (does the model think it was mispronounced?)

Insertions (phones in gold/ASR with no canonical counterpart) are excluded from evaluation
since they don't correspond to a canonical position to accept or reject.

| | Model: mispronounced | Model: correct |
|---|---|---|
| **Actually mispronounced** | True Rejection (TR) | False Acceptance (FA) |
| **Actually correct** | False Rejection (FR) | True Acceptance (TA) |

# Smoke Test

In [11]:
# Canonical:  b  a  d
# Gold:       p  a  d   → L2 speaker substituted b→p  (1 mispronunciation)
# ASR output: p  a  t   → model caught b→p, but also flagged d→t (1 TR, 1 FR)
#
# Expected:
#   position 0 (b): gt=True,  pred=True  → TR
#   position 1 (a): gt=False, pred=False → TA
#   position 2 (d): gt=False, pred=True  → FR

_canonical = ['b', 'a', 'd']
_gold      = ['p', 'a', 'd']
_asr       = ['p', 'a', 't']

result = evaluate_mispronunciation_detection(_canonical, _gold, _asr)
assert result['TR'] == 1
assert result['TA'] == 1
assert result['FR'] == 1
assert result['FA'] == 0
print("Smoke test passed:", result)

Smoke test passed: {'TR': 1, 'FA': 0, 'FR': 1, 'TA': 1, 'precision': 0.5, 'recall': 1.0, 'f1': 0.6666666666666666, 'false_acceptance_rate': 0.0, 'false_rejection_rate': 0.5, 'per': 0.6666666666666666}
